# **Etapa 1: Conceitos e Configuração do Ambiente Virtual**

## **1. Diferença entre LLM e Agente de IA**

Um **Modelo de Linguagem (LLM)** puro — como o GPT-4o ou modelos Llama — é um **motor preditivo de texto**. Ele recebe um contexto de entrada (*prompt*) e calcula qual é a próxima palavra estatisticamente mais provável. Ele não executa ações por conta própria, não acessa arquivos locais do seu computador e não realiza buscas externas sem que uma infraestrutura externa o invoque.

Já um **Agente de IA** é um sistema autônomo construído tendo o LLM como cérebro de raciocínio, amparado por três pilares fundamentais:

1. **Objetivo (*Goal*):** Uma meta clara definida para o sistema (ex: *"Analisar o relatório em PDF e responder às dúvidas do usuário com precisão"*).
2. **Plano (*Planning*):** A capacidade de raciocinar em etapas, dividindo um problema complexo em sub-tarefas.
3. **Ferramentas (*Tools*):** A capacidade de invocar funções externas proativamente (ex: consultar o banco vetorial **ChromaDB**, ler arquivos com **pypdf** ou chamar APIs REST em **Flask**).

---

## **2. Passo a Passo de Preparação do Ambiente (`.venv`)**

Para garantir compatibilidade total com a biblioteca **CrewAI** e evitar conflitos entre dependências, utilizaremos o **Python 3.13** dentro de um ambiente virtual isolado.

### **Comandos de Execução no Terminal:**

1. **Instalar o Python 3.13** (caso ainda não esteja instalado no sistema):
   ```bash
   winget install Python.Python.3.13
   ```

2. **Criar o Ambiente Virtual (`.venv`):**
   ```bash
   py -3.13 -m venv .venv
   ```

3. **Ativar o Ambiente Virtual (Windows - PowerShell):**
   ```powershell
   Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
   .\.venv\Scripts\Activate
   ```

4. **Atualizar o Gerenciador de Pacotes (`pip`):**
   ```bash
   python -m pip install --upgrade pip
   ```

# **Etapa 2: Instalação de Pacotes, Variáveis de Ambiente e Conexão (`src/config.py`)**

## **1. Instalação das Bibliotecas do Projeto**

Com o ambiente virtual (`.venv`) ativo, executamos a instalação dos pacotes necessários para o **CrewAI**, **LangChain**, **Azure OpenAI**, **ChromaDB** e **Flask**.

### **Comando de Execução no Terminal:**

```bash
pip install python-dotenv crewai "crewai[azure-ai-inference]" langchain-openai pypdf fontTools langchain langchain-text-splitters chromadb langchain-chroma flask werkzeug
```

---

## **2. Configuração de Segurança com o Arquivo `.env`**

Nunca salvamos chaves de API ou credenciais privadas diretamente no código-fonte. Criamos um arquivo chamado `.env` na raiz do projeto para armazenar estas variáveis com segurança.

### **Conteúdo do Arquivo `.env`:**

In [ ]:
AZURE_OPENAI_API_KEY=aqui_entra_a_sua_chave
AZURE_OPENAI_ENDPOINT=[https://seu-recurso.openai.azure.com/](https://seu-recurso.openai.azure.com/)
AZURE_OPENAI_API_VERSION=2024-02-15-preview
AZURE_CHAT_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_EMBEDDINGS_DEPLOYMENT_NAME=text-embedding-3-small

## **3. Criação do Módulo de Conexão (`src/config.py`)**

Este módulo lê o arquivo `.env` e inicializa as instâncias de comunicação com a Azure OpenAI e o CrewAI.

Crie a pasta `src/` na raiz do projeto e, dentro dela, o arquivo `config.py`:

### **Código Python (`src/config.py`):**

In [ ]:
import os # 1. para acessar variáveis de ambiente do sistema operacional
from dotenv import load_dotenv # 1. para carregar variáveis de ambiente de forma segura
from crewai import LLM # 2. orquestração de agentes autônomos, para definir agentes, tarefas e fluxos de execução (crews), além de incluir a classe LLM
from langchain_openai import AzureOpenAIEmbeddings # 2. voltado para integração com os modelos da OpenAI e da Azure OpenAI

# 1. Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# 2. Configura o modelo LLM da Azure para uso com o CrewAI
azure_llm = LLM(
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

# 3. Configura o modelo de Embeddings da Azure OpenAI para o ChromaDB
azure_embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

print("Conexão com Azure OpenAI e CrewAI configurada com sucesso!")

# **Etapa 3: Leitura de PDF, Fatiamento (Chunking) e Banco Vetorial (`src/tools.py`)**

## **1. Entendendo o RAG (Retrieval-Augmented Generation)**

Modelos de linguagem não possuem acesso nativo a documentos locais do seu computador. Para permitir que a IA responda a dúvidas com base em um documento específico (como um relatório em PDF), utilizamos a arquitetura **RAG**:

1. **Leitura do PDF:** Extraímos o texto bruto do documento usando a biblioteca `pypdf`.
2. **Chunking (Fatiamento):** Dividimos o texto completo em blocos menores (*chunks*) para facilitar a busca e respeitar o limite de contexto do modelo.
3. **Embeddings:** Convertemos cada bloco de texto em vetores numéricos que representam o seu significado semântico.
4. **Persistência Vetorial:** Armazenamos esses vetores no **ChromaDB** para buscas rápidas por similaridade.
5. **Ferramenta (`@tool`):** Encapsulamos a lógica de consulta em uma função registrada com o decorador `@tool`, permitindo que os agentes do **CrewAI** invoquem essa busca proativamente.

---

## **2. Criação do Módulo de Processamento e Busca (`src/tools.py`)**

Crie o arquivo `tools.py` dentro do diretório `src/`:

### **Código Python (`src/tools.py`):**

In [ ]:
import os
import shutil
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from src.config import azure_embeddings
from langchain_core.tools import tool

PASTA_PDFS = "pdfs"
DIRETORIO_DB = "chroma_db"

def processar_pdf():
    """Limpa e recria o banco vetorial no ChromaDB a partir dos PDFs na pasta 'pdfs/'."""
    os.makedirs(PASTA_PDFS, exist_ok=True)

    # 1. Apaga a pasta chroma_db antiga para eliminar vetores de PDFs removidos
    if os.path.exists(DIRETORIO_DB):
        try:
            shutil.rmtree(DIRETORIO_DB)
            print(f"Banco antigo em '{DIRETORIO_DB}' resetado para re-indexação.")
        except Exception as e:
            print(f"Aviso ao limpar diretório do banco: {e}")

    # 2. Busca todos os PDFs atualmente na pasta pdfs/
    arquivos_pdf = [os.path.join(PASTA_PDFS, f) for f in os.listdir(PASTA_PDFS) if f.lower().endswith('.pdf')]

    # Caso a pasta esteja vazia, encerra informando a limpeza
    if not arquivos_pdf:
        print("Nenhum arquivo PDF restou na pasta. O banco vetorial está limpo.")
        return None

    textos_paginas = []
    metadados = []

    for caminho_pdf in arquivos_pdf:
        nome_arquivo = os.path.basename(caminho_pdf)
        print(f"Processando: {nome_arquivo}...")
        leitor = PdfReader(caminho_pdf)
        for pagina in leitor.pages:
            texto_pagina = pagina.extract_text()
            if texto_pagina:
                textos_paginas.append(texto_pagina)
                metadados.append({"source": nome_arquivo})

    if not textos_paginas:
        print("Nenhum texto válido foi extraído dos PDFs restantes.")
        return None

    # 3. Fatiamento do texto em chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    docs = text_splitter.create_documents(textos_paginas, metadatas=metadados)
    print(f"Total de {len(docs)} chunks gerados.")

    # 4. Geração dos novos embeddings e persistência
    print("Gerando novos embeddings no ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=azure_embeddings,
        persist_directory=DIRETORIO_DB
    )
    print("Banco vetorial sincronizado com sucesso!")
    return vectorstore


@tool("Consultar Base de Conhecimento PDF")
def consultar_base_conhecimento(query: str) -> str:
    """Útil para consultar informações presentes nos documentos PDF cadastrados."""
    if not os.path.exists(DIRETORIO_DB):
        return "Nenhum documento cadastrado na base de conhecimento no momento."

    vectorstore = Chroma(
        persist_directory=DIRETORIO_DB,
        embedding_function=azure_embeddings
    )

    resultados = vectorstore.similarity_search(query, k=3)

    if not resultados:
        return "Nenhuma informação relevante foi encontrada na base de conhecimento."

    return "\n\n---\n\n".join([doc.page_content for doc in resultados])


if __name__ == "__main__":
    processar_pdf()

# **Etapa 4: Servidor Web e API RESTful (`app.py`)**

## **1. Entendendo a Arquitetura Backend com Flask**

O **Flask** é um microframework web em Python responsável por disponibilizar endpoints (rotas HTTP) que integram a lógica de Inteligência Artificial à interface gráfica do utilizador.

---

### **Mapeamento das Rotas do Servidor:**

* **`GET /` (`index`):** Renderiza e entrega a página HTML principal do chat (`templates/index.html`).
* **`GET /api/pdfs` (`listar_pdfs`):** Retorna uma lista em formato JSON com os arquivos PDF presentes na pasta de uploads.
* **`POST /perguntar` (`perguntar`):** Recebe a dúvida digitada pelo usuário via JSON, invoca a ferramenta RAG (`consultar_base_conhecimento`) e devolve a resposta gerada.
* **`POST /upload` (`upload_pdf`):** Recebe um novo arquivo PDF enviado pela interface, sanitiza o seu nome usando `secure_filename` e o armazena na pasta `pdfs/`.
* **`DELETE /remover/<filename>` (`remover_pdf`):** Remove fisicamente o arquivo PDF selecionado do diretório do servidor.

---

## **2. Criação do Servidor Principal (`app.py`)**

Crie o arquivo `app.py` na raiz do seu projeto:

### **Código Python (`app.py`):**

In [ ]:
import os
from flask import Flask, render_template, request, jsonify
from werkzeug.utils import secure_filename
from src.tools import processar_pdf, consultar_base_conhecimento

# 1. Inicialização da aplicação Flask
app = Flask(__name__)
app.secret_key = 'sua_chave_secreta_aqui'

# 2. Configuração da pasta onde os arquivos PDF serão armazenados
UPLOAD_FOLDER = os.path.join(os.getcwd(), 'pdfs')
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# Restrição de extensões permitidas
ALLOWED_EXTENSIONS = {'pdf'}

def allowed_file(filename):
    """Verifica se o arquivo enviado possui a extensão .pdf"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


@app.route("/")
def index():
    """Rota principal: carrega a interface gráfica do chatbot"""
    return render_template("index.html")


@app.route("/api/pdfs", methods=["GET"])
def listar_pdfs():
    """Endpoint API: Retorna a lista de PDFs armazenados no servidor"""
    try:
        arquivos = os.listdir(app.config['UPLOAD_FOLDER'])
        pdfs = [f for f in arquivos if f.lower().endswith('.pdf')]
        return jsonify({"status": "sucesso", "arquivos": pdfs})
    except Exception as e:
        return jsonify({"status": "erro", "arquivos": [], "mensagem": str(e)})


@app.route("/perguntar", methods=["POST"])
def perguntar():
    """Endpoint API: Recebe a pergunta do usuário e executa a busca RAG"""
    dados = request.get_json()
    pergunta_usuario = dados.get("pergunta", "")
    
    if not pergunta_usuario.strip():
        return jsonify({"resposta": "Por favor, digite uma pergunta válida."})
    
    try:
        # Invocação da ferramenta de busca RAG criada em src/tools.py
        resposta = consultar_base_conhecimento.invoke(pergunta_usuario)
        return jsonify({"resposta": resposta})
    except Exception as e:
        return jsonify({"resposta": f"Erro ao processar a requisição: {str(e)}"}), 500


@app.route("/upload", methods=["POST"])
def upload_pdf():
    """Endpoint API: Recebe, salva e vetoriza automaticamente um novo PDF"""
    if 'file' not in request.files:
        return jsonify({"status": "erro", "mensagem": "Nenhum arquivo enviado."}), 400
    
    file = request.files['file']
    
    if file.filename == '':
        return jsonify({"status": "erro", "mensagem": "Nenhum arquivo selecionado."}), 400
    
    if file and allowed_file(file.filename):
        filename = secure_filename(file.filename)
        caminho_salvamento = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(caminho_salvamento)
        
        # Processa o PDF e atualiza a base no ChromaDB imediatamente
        try:
            processar_pdf()
            return jsonify({
                "status": "sucesso", 
                "mensagem": f"PDF '{filename}' enviado e vetorizado no ChromaDB com sucesso!"
            })
        except Exception as e:
            return jsonify({
                "status": "erro", 
                "mensagem": f"PDF salvo em disco, mas ocorreu um erro na vetorização: {str(e)}"
            }), 500
    else:
        return jsonify({"status": "erro", "mensagem": "Apenas arquivos PDF são permitidos."}), 400


@app.route("/remover/<filename>", methods=["DELETE"])
def remover_pdf(filename):
    """Endpoint API: Exclui o arquivo PDF e ressincroniza o banco vetorial"""
    filename = secure_filename(filename)
    caminho_arquivo = os.path.join(app.config['UPLOAD_FOLDER'], filename)

    if os.path.exists(caminho_arquivo):
        try:
            # 1. Apaga o arquivo físico da pasta pdfs/
            os.remove(caminho_arquivo)

            # 2. Recria a base vetorial sem o arquivo apagado
            processar_pdf()

            return jsonify({
                "status": "sucesso",
                "mensagem": f"PDF '{filename}' removido e base de conhecimento atualizada!"
            })
        except Exception as e:
            return jsonify({"status": "erro", "mensagem": f"Erro ao excluir arquivo: {str(e)}"}), 500
    else:
        return jsonify({"status": "erro", "mensagem": "Arquivo não encontrado."}), 404


if __name__ == "__main__":
    app.run(debug=True, port=5000)

# **Etapa 5 (Aprimorada): Interface Gráfica Profissional em Dark Mode**

## **1. Melhorias de Design e Usabilidade (UI/UX)**

Para transformar o visual do projeto em uma aplicação moderna e atraente (nível de software comercial), implementamos as seguintes melhorias:

1. **Layout em Duas Colunas (Sidebar + Chat):** Adicionamos uma barra lateral dedicada ao gerenciamento de PDFs (upload, listagem e exclusão), aproveitando os endpoints criados na **Etapa 4**.
2. **Tipografia Moderna (Google Fonts):** Integração da fonte *Inter* para leitura clara.
3. **Indicadores Visuais de Status:** Ponto verde indicador de sistema *Online* e animações de entrada de mensagens (*Fade In*).
4. **Ícones Vetoriais SVG:** Substituição de botões simples por ícones minimalistas para envio, upload e exclusão.
5. **Scrollbar Personalizada e Efeitos Hover:** Microinterações visuais com transições suaves.

---

## **2. Código HTML Atualizado (`templates/index.html`)**

Substitua o arquivo `templates/index.html` pelo código abaixo:

### **Código HTML (`templates/index.html`):**

In [ ]:
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Assistente IA - Base de Conhecimento RAG</title>
    <link rel="preconnect" href="[https://fonts.googleapis.com](https://fonts.googleapis.com)">
    <link rel="preconnect" href="[https://fonts.gstatic.com](https://fonts.gstatic.com)" crossorigin>
    <link href="[https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap](https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap)" rel="stylesheet">
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <div class="app-container">
        <!-- Barra Lateral: Gerenciamento de PDFs -->
        <aside class="sidebar">
            <div class="sidebar-header">
                <h2> Documentos PDF</h2>
                <p>Base de conhecimento ativa</p>
            </div>

            <!-- Formulário de Upload -->
            <div class="upload-section">
                <label for="fileInput" class="upload-btn">
                    <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"/><polyline points="17 8 12 3 7 8"/><line x1="12" y1="3" x2="12" y2="15"/></svg>
                    Carregar Novo PDF
                </label>
                <input type="file" id="fileInput" accept=".pdf" onchange="fazerUploadPDF()" hidden>
            </div>

            <!-- Lista de Arquivos Cadastrados -->
            <div class="pdf-list-container">
                <h3>Arquivos Disponíveis</h3>
                <ul id="pdfList" class="pdf-list">
                    <li class="loading-item">Carregando lista...</li>
                </ul>
            </div>
        </aside>

        <!-- Área Principal do Chatbot -->
        <main class="chat-area">
            <header class="chat-header">
                <div class="agent-info">
                    <div class="status-badge"></div>
                    <div>
                        <h1>Agente de IA RAG</h1>
                        <p class="subtitle">Conectado ao ChromaDB & Azure OpenAI</p>
                    </div>
                </div>
            </header>

            <div class="chat-messages" id="chatMessages">
                <div class="message bot">
                    <div class="avatar">🤖</div>
                    <div class="bubble">
                        Olá! Sou o seu assistente virtual de inteligência artificial. Como posso ajudar com base nos documentos cadastrados?
                    </div>
                </div>
            </div>

            <div class="chat-input-area">
                <input type="text" id="userInput" placeholder="Digite sua pergunta sobre os documentos..." onkeypress="handleKeyPress(event)">
                <button onclick="enviarPergunta()" id="btnSend">
                    <span>Enviar</span>
                    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><line x1="22" y1="2" x2="11" y2="13"/><polygon points="22 2 15 22 11 13 2 9 22 2"/></svg>
                </button>
            </div>
        </main>
    </div>

    <script>
        // Carrega a lista de PDFs ao abrir a página
        document.addEventListener('DOMContentLoaded', carregarPDFs);

        async function carregarPDFs() {
            const pdfList = document.getElementById('pdfList');
            try {
                const res = await fetch('/api/pdfs');
                const data = await res.json();
                
                pdfList.innerHTML = '';
                if (data.arquivos.length === 0) {
                    pdfList.innerHTML = '<li class="empty-item">Nenhum PDF cadastrado.</li>';
                    return;
                }

                data.arquivos.forEach(pdf => {
                    const li = document.createElement('li');
                    li.className = 'pdf-item';
                    li.innerHTML = `
                        <span class="pdf-name" title="${pdf}">📄 ${pdf}</span>
                        <button class="btn-delete" onclick="removerPDF('${pdf}')" title="Excluir">🗑️</button>
                    `;
                    pdfList.appendChild(li);
                });
            } catch (err) {
                pdfList.innerHTML = '<li class="error-item">Erro ao carregar PDFs.</li>';
            }
        }

        async function fazerUploadPDF() {
            const fileInput = document.getElementById('fileInput');
            if (!fileInput.files[0]) return;

            const formData = new FormData();
            formData.append('file', fileInput.files[0]);

            try {
                const res = await fetch('/upload', { method: 'POST', body: formData });
                const data = await res.json();
                alert(data.mensagem);
                fileInput.value = '';
                carregarPDFs();
            } catch (err) {
                alert('Erro ao enviar o arquivo.');
            }
        }

        async function removerPDF(filename) {
            if (!confirm(`Deseja remover o arquivo '${filename}'?`)) return;

            try {
                const res = await fetch(`/remover/${filename}`, { method: 'DELETE' });
                const data = await res.json();
                alert(data.mensagem);
                carregarPDFs();
            } catch (err) {
                alert('Erro ao excluir o arquivo.');
            }
        }

        async function enviarPergunta() {
            const input = document.getElementById('userInput');
            const pergunta = input.value.trim();
            if (!pergunta) return;

            const chatMessages = document.getElementById('chatMessages');

            // Mensagem do Usuário
            const userDiv = document.createElement('div');
            userDiv.className = 'message user';
            userDiv.innerHTML = `
                <div class="bubble">${escapeHtml(pergunta)}</div>
                <div class="avatar">👤</div>
            `;
            chatMessages.appendChild(userDiv);

            input.value = '';
            chatMessages.scrollTop = chatMessages.scrollHeight;

            // Mensagem de Processamento do Bot
            const botDiv = document.createElement('div');
            botDiv.className = 'message bot';
            botDiv.innerHTML = `
                <div class="avatar">🤖</div>
                <div class="bubble loading">Consultando base de conhecimento...</div>
            `;
            chatMessages.appendChild(botDiv);
            chatMessages.scrollTop = chatMessages.scrollHeight;

            try {
                const response = await fetch('/perguntar', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ pergunta: pergunta })
                });
                const data = await response.json();
                botDiv.querySelector('.bubble').classList.remove('loading');
                botDiv.querySelector('.bubble').textContent = data.resposta;
            } catch (error) {
                botDiv.querySelector('.bubble').classList.remove('loading');
                botDiv.querySelector('.bubble').textContent = 'Erro ao processar a requisição no servidor.';
            }

            chatMessages.scrollTop = chatMessages.scrollHeight;
        }

        function handleKeyPress(event) {
            if (event.key === 'Enter') enviarPergunta();
        }

        function escapeHtml(text) {
            return text.replace(/[&<>"']/g, function(m) {
                return { '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#039;' }[m];
            });
        }
    </script>
</body>
</html>

## **3. Código CSS Moderno (`static/style.css`)**

Substitua o arquivo `static/style.css` pelo estilo refinado em Dark Mode abaixo:

### **Código CSS (`static/style.css`):**

In [ ]:
:root {
    --bg-dark: #090d16;
    --sidebar-bg: #111827;
    --chat-bg: #1e293b;
    --card-bg: #1f2937;
    --accent-color: #2563eb;
    --accent-hover: #1d4ed8;
    --text-primary: #f8fafc;
    --text-secondary: #94a3b8;
    --border-color: #334155;
    --user-bubble: #2563eb;
    --bot-bubble: #334155;
}

* {
    box-sizing: border-box;
}

body {
    font-family: 'Inter', system-ui, -apple-system, sans-serif;
    background-color: var(--bg-dark);
    color: var(--text-primary);
    margin: 0;
    padding: 0;
    height: 100vh;
    display: flex;
    justify-content: center;
    align-items: center;
}

/* Layout Principal */
.app-container {
    width: 95vw;
    max-width: 1200px;
    height: 90vh;
    background-color: var(--sidebar-bg);
    border-radius: 16px;
    border: 1px solid var(--border-color);
    box-shadow: 0 20px 50px rgba(0, 0, 0, 0.5);
    display: flex;
    overflow: hidden;
}

/* Barra Lateral */
.sidebar {
    width: 320px;
    background-color: var(--sidebar-bg);
    border-right: 1px solid var(--border-color);
    display: flex;
    flex-direction: column;
    padding: 24px;
}

.sidebar-header h2 {
    margin: 0;
    font-size: 1.15rem;
    color: #60a5fa;
}

.sidebar-header p {
    margin: 4px 0 20px 0;
    font-size: 0.85rem;
    color: var(--text-secondary);
}

.upload-btn {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 8px;
    background-color: var(--accent-color);
    color: white;
    padding: 12px;
    border-radius: 8px;
    font-weight: 600;
    font-size: 0.9rem;
    cursor: pointer;
    transition: background 0.2s, transform 0.1s;
}

.upload-btn:hover {
    background-color: var(--accent-hover);
    transform: translateY(-1px);
}

.pdf-list-container {
    margin-top: 24px;
    flex: 1;
    overflow-y: auto;
}

.pdf-list-container h3 {
    font-size: 0.85rem;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    color: var(--text-secondary);
    margin-bottom: 12px;
}

.pdf-list {
    list-style: none;
    padding: 0;
    margin: 0;
    display: flex;
    flex-direction: column;
    gap: 8px;
}

.pdf-item {
    display: flex;
    align-items: center;
    justify-content: space-between;
    background-color: var(--card-bg);
    padding: 10px 12px;
    border-radius: 6px;
    border: 1px solid var(--border-color);
    font-size: 0.85rem;
}

.pdf-name {
    overflow: hidden;
    text-overflow: ellipsis;
    white-space: nowrap;
    max-width: 200px;
}

.btn-delete {
    background: none;
    border: none;
    cursor: pointer;
    font-size: 0.9rem;
    opacity: 0.7;
    transition: opacity 0.2s;
}

.btn-delete:hover {
    opacity: 1;
}

/* Área Principal do Chat */
.chat-area {
    flex: 1;
    display: flex;
    flex-direction: column;
    background-color: var(--chat-bg);
}

.chat-header {
    padding: 18px 24px;
    background-color: #111827;
    border-bottom: 1px solid var(--border-color);
}

.agent-info {
    display: flex;
    align-items: center;
    gap: 12px;
}

.status-badge {
    width: 10px;
    height: 10px;
    background-color: #10b981;
    border-radius: 50%;
    box-shadow: 0 0 8px #10b981;
}

.chat-header h1 {
    margin: 0;
    font-size: 1.1rem;
}

.subtitle {
    margin: 2px 0 0 0;
    font-size: 0.8rem;
    color: var(--text-secondary);
}

/* Histórico de Mensagens */
.chat-messages {
    flex: 1;
    padding: 24px;
    overflow-y: auto;
    display: flex;
    flex-direction: column;
    gap: 18px;
}

.message {
    display: flex;
    align-items: flex-end;
    gap: 10px;
    max-width: 80%;
    animation: fadeIn 0.3s ease-out;
}

.message.bot {
    align-self: flex-start;
}

.message.user {
    align-self: flex-end;
    flex-direction: row;
}

.avatar {
    width: 36px;
    height: 36px;
    border-radius: 50%;
    background-color: #334155;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 1.1rem;
    flex-shrink: 0;
}

.bubble {
    padding: 12px 16px;
    border-radius: 12px;
    font-size: 0.95rem;
    line-height: 1.5;
    white-space: pre-wrap;
}

.message.bot .bubble {
    background-color: var(--bot-bubble);
    color: var(--text-primary);
    border-bottom-left-radius: 2px;
}

.message.user .bubble {
    background-color: var(--user-bubble);
    color: white;
    border-bottom-right-radius: 2px;
}

.bubble.loading {
    font-style: italic;
    opacity: 0.8;
}

/* Área de Entrada de Texto */
.chat-input-area {
    padding: 20px 24px;
    background-color: #111827;
    border-top: 1px solid var(--border-color);
    display: flex;
    gap: 12px;
}

.chat-input-area input {
    flex: 1;
    padding: 12px 16px;
    border-radius: 8px;
    border: 1px solid var(--border-color);
    background-color: var(--chat-bg);
    color: white;
    font-size: 0.95rem;
}

.chat-input-area input:focus {
    outline: none;
    border-color: var(--accent-color);
}

.chat-input-area button {
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 0 20px;
    background-color: var(--accent-color);
    color: white;
    border: none;
    border-radius: 8px;
    font-weight: 600;
    cursor: pointer;
    transition: background-color 0.2s;
}

.chat-input-area button:hover {
    background-color: var(--accent-hover);
}

@keyframes fadeIn {
    from { opacity: 0; transform: translateY(6px); }
    to { opacity: 1; transform: translateY(0); }
}

/* Scrollbar Personalizada */
::-webkit-scrollbar {
    width: 6px;
}
::-webkit-scrollbar-track {
    background: transparent;
}
::-webkit-scrollbar-thumb {
    background: #334155;
    border-radius: 3px;
}

# **Etapa 6: Criação do `requirements.txt`, Estrutura do Projeto e Execução**

## **1. Gerando e Otimizando o Arquivo `requirements.txt`**

Para que os alunos e outros desenvolvedores possam reexecutar o projeto em qualquer máquina, precisamos congelar e exportar as dependências instaladas.

### **Passo a Passo no Terminal:**

1. **Gerar a lista automática de dependências ativas:**
   ```bash
   pip freeze > requirements.txt
   ```

2. **Limpeza do arquivo (Recomendado):**
   Abertura do arquivo `requirements.txt` gerado para manter apenas as bibliotecas principais do projeto, garantindo compatibilidade e evitando conflitos de pacotes transitivos.

### **Conteúdo Final Otimizado do `requirements.txt`:**

```text
flask==3.1.0
werkzeug==3.1.3
python-dotenv==1.0.1
pypdf==5.1.0
fontTools==4.55.3
langchain==0.3.14
langchain-core==0.3.29
langchain-community==0.3.14
langchain-openai==0.3.0
langchain-chroma==0.2.0
langchain-text-splitters==0.3.5
crewai==0.102.0
chromadb==0.6.3
```

---

## **2. Estrutura Final de Arquivos e Pastas**

Confira se a estrutura do seu workspace está exatamente assim antes da execução:

```text
meu_projeto_rag/
│
├── .env                          # Variáveis de ambiente e chaves da Azure OpenAI
├── .venv/                        # Ambiente virtual isolado (Python 3.13)
├── app.py                        # Servidor principal Flask (Rotas HTTP e API REST)
├── requirements.txt              # Dependências homologadas do projeto
├── plano-enf-doencas-2022_2030.pdf# Arquivo PDF base para consulta RAG
│
├── pdfs/                         # Pasta local para armazenamento dos uploads
├── chroma_db/                    # Banco de dados vetorial criado pelo ChromaDB
│
├── src/                          # Módulos Python do sistema
│   ├── config.py                 # Conexão com Azure OpenAI e CrewAI
│   └── tools.py                  # Processamento de PDF, Embeddings e Tool RAG
│
├── static/                       # Arquivos estáticos
│   └── style.css                 # Estilização CSS em Dark Mode
│
└── templates/                    # Telas em HTML
    └── index.html                # Interface gráfica com barra lateral e chat
```

---

## **3. Passo a Passo para Testar e Executar a Aplicação**

Com o ambiente virtual `.venv` ativo no terminal, siga a ordem de execução:

### **Passo 1: Criar a Base Vetorial no ChromaDB**
Execute o script `tools.py` para realizar a leitura do PDF inicial e popular a base vetorial:

```bash
python -m src.tools
```

### **Passo 2: Iniciar o Servidor Flask**
Suba a aplicação web executando o servidor principal:

```bash
python app.py
```

### **Passo 3: Abrir no Navegador**
Acesse o endereço exibido no terminal para interagir com o agente de IA:

```text
[http://127.0.0.1:5000](http://127.0.0.1:5000)
```